### Setting

In [1]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [2]:
import os
import pandas as pd

import torch
import torch.nn as nn

#import gensim
#from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

import re
import numpy as np

from sklearn.preprocessing import LabelEncoder

import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import torch.nn.functional as F

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from collections import Counter

from transformers import BertTokenizer, BertModel

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available!")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

GPU is available!


### Data Loader

In [4]:
cd gdrive/MyDrive/Master degree/24T3/DATA5011/modelling_final/Datasets/ver8_10k

/content/gdrive/MyDrive/Master degree/24T3/DATA5011/modelling_final/Datasets/ver8_10k


In [5]:
I_plus_train = pd.read_parquet('I_plus_train_10k_ver8.parquet')
I_minus_train = pd.read_parquet('I_minus_train_10k_ver8.parquet')

I_plus_val = pd.read_parquet('I_plus_val_10k_ver8.parquet')
I_minus_val = pd.read_parquet('I_minus_val_10k_ver8.parquet')

test = pd.read_parquet('test_10k_ver8.parquet')

In [6]:
I_plus_train

,pid,track_id,track_name,album_name,artist_name,danceability,energy,key,loudness,mode,...,song_count,lemmatized_track_name,lemmatized_album_name,track_sentences,album_sentences,artist_id,audio_features,track_embeddings,album_embeddings,new_pid
0,3,39055,Just Get High,Die Alone,Gazebos,0.337,0.701,9.0,-6.974,1.0,...,123,get high,die alone,"[get, high]","[die, alone]",7899,"[0.337, 0.701, 9.0, -6.974, 1.0, 0.0306, 0.001...","[[0.033203125, -0.08984375, -0.294921875, 0.11...","[[0.10009765625, 0.17578125, 0.043701171875, 0...",0
1,3,52439,Nothing's Gonna Hurt You Baby,I.,Cigarettes After Sex,0.509,0.331,4.0,-14.083,1.0,...,123,nothing gonna hurt baby,i,"[nothing, gonna, hurt, baby]",[i],4150,"[0.509, 0.331, 4.0, -14.083, 1.0, 0.0267, 0.27...","[[0.1640625, 0.056396484375, 0.08251953125, -0...","[[-0.2255859375, -0.01953125, 0.0908203125, 0....",0
2,3,33924,I Like You,It's All In Your Head,dandelion hands,0.350,0.100,4.0,-18.491,1.0,...,123,like,head,[like],[head],24118,"[0.35, 0.1, 4.0, -18.491, 1.0, 0.0328, 0.989, ...","[[0.103515625, 0.1376953125, -0.00297546386718...","[[-0.0712890625, -0.07373046875, 0.19921875, -...",0
3,3,15639,Crystal,Candy Apple Grey,Hüsker Dü,0.386,0.964,7.0,-11.310,1.0,...,123,crystal,candy apple grey,[crystal],"[candy, apple, grey]",9198,"[0.386, 0.964, 7.0, -11.31, 1.0, 0.076, 0.0589...","[[0.045654296875, -0.26953125, 0.0732421875, -...","[[-0.050048828125, -0.232421875, -0.0308837890...",0
4,3,85903,You Say I'm in Love,You Say I'm in Love,Banes World,0.625,0.285,1.0,-16.686,0.0,...,123,say im love,say im love,"[say, im, love]","[say, im, love]",1905,"[0.625, 0.285, 1.0, -16.686, 0.0, 0.0537, 0.43...","[[-0.0361328125, -0.12109375, 0.1337890625, 0....","[[-0.0361328125, -0.12109375, 0.1337890625, 0....",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7065,453,36170,If You're Gone,Mad Season,Matchbox Twenty,0.544,0.659,9.0,-7.191,1.0,...,155,youre go,mad season,"[youre, go]","[mad, season]",13932,"[0.544, 0.659, 9.0, -7.191, 1.0, 0.0298, 0.427...","[[0.28125, 0.033447265625, -0.035888671875, 0....","[[0.055419921875, -0.056640625, 0.150390625, 0...",100
7066,453,6723,Barely Breathing,Duncan Sheik,Duncan Sheik,0.482,0.807,0.0,-6.976,1.0,...,100,barely breathing,duncan sheik,"[barely, breathing]","[duncan, sheik]",6250,"[0.482, 0.807, 0.0, -6.976, 1.0, 0.0457, 0.028...","[[0.2021484375, -0.25390625, -0.035400390625, ...","[[-0.06005859375, 0.1591796875, 0.019287109375...",100
7067,453,45970,Making Love Out of Nothing at All,The Best of Air Supply: Ones That You Love,Air Supply,0.386,0.701,7.0,-5.993,1.0,...,155,make love nothing,best air supply one love,"[make, love, nothing]","[best, air, supply, one, love]",574,"[0.386, 0.701, 7.0, -5.993, 1.0, 0.0332, 0.14,...","[[-0.11328125, -0.036865234375, 0.09423828125,...","[[-0.126953125, 0.02197265625, 0.287109375, 0....",100
7068,453,32266,How Deep Is Your Love,Greatest,Bee Gees,0.633,0.357,5.0,-9.366,0.0,...,175,deep love,great,"[deep, love]",[great],2071,"[0.633, 0.357, 5.0, -9.366, 0.0, 0.0264, 0.105...","[[-0.0361328125, 0.1494140625, 0.1376953125, 0...","[[0.07177734375, 0.2080078125, -0.028442382812...",100


In [ ]:
# Initialize the BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Make sure the model is in evaluation mode
model.eval()

# Convert the 'TrackArtist' column into BERT embeddings
def get_bert_embedding(text):
    # Tokenize and encode the input text
    #inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=10, padding=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=32, padding=True)

    # Get the [CLS] token embeddings
    with torch.no_grad():  # Disable gradient calculation for efficiency
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # Extract the [CLS] token representation

    return cls_embedding.squeeze().numpy()  # Convert to numpy for easier storage

In [8]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['album_Bert_embedding'] = I_plus_train['album_name'].apply(get_bert_embedding)
I_minus_train['album_Bert_embedding'] = I_minus_train['album_name'].apply(get_bert_embedding)
I_plus_val['album_Bert_embedding'] = I_plus_val['album_name'].apply(get_bert_embedding)
I_minus_val['album_Bert_embedding'] = I_minus_val['album_name'].apply(get_bert_embedding)
test['album_Bert_embedding'] = test['album_name'].apply(get_bert_embedding)
#music_pool['album_Bert_embedding'] = music_pool['lemmatized_album_name'].apply(get_bert_embedding)

In [9]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['track_Bert_embedding'] = I_plus_train['track_name'].apply(get_bert_embedding)
I_minus_train['track_Bert_embedding'] = I_minus_train['track_name'].apply(get_bert_embedding)
I_plus_val['track_Bert_embedding'] = I_plus_val['track_name'].apply(get_bert_embedding)
I_minus_val['track_Bert_embedding'] = I_minus_val['track_name'].apply(get_bert_embedding)
test['track_Bert_embedding'] = test['track_name'].apply(get_bert_embedding)
#music_pool['track_Bert_embedding'] = music_pool['lemmatized_track_name'].apply(get_bert_embedding)

In [24]:
test

,pid,track_name,album_name,artist_name,danceability,energy,key,loudness,mode,speechiness,...,track_sentences,album_sentences,track_id,artist_id,audio_features,track_embeddings,album_embeddings,new_pid,album_Bert_embedding,track_Bert_embedding
0,3,i was all over her,melanchole,salvia palth,0.529,0.353,7.0,-12.835,1.0,0.0292,...,"[i, be, all, over, her]",[melanchole],86927,24214,"[0.529, 0.353, 7.0, -12.835, 1.0, 0.0292, 0.78...","[[-0.2255859375, -0.01953125, 0.0908203125, 0....","[[-0.07382528483867645, -0.6845917105674744, 0...",0,"[-0.40084502, -0.15834248, -0.20055139, -0.413...","[-0.040514406, 0.0137907425, -0.07234939, 0.06..."
1,3,Kalte Wut / Wenn Ich Einmal Reich Bin,Lucky Streik,Floh de Cologne,0.514,0.733,7.0,-9.443,1.0,0.1230,...,"[kalte, wut, wenn, ich, einmal, reich, bin]","[lucky, streik]",39296,7364,"[0.514, 0.733, 7.0, -9.443, 1.0, 0.123, 0.545,...","[[1.4862676858901978, -0.315891295671463, 1.53...","[[0.05029296875, -0.0308837890625, -0.09912109...",0,"[-0.24293439, 0.22737335, 0.046346705, 0.10388...","[-0.47904563, 0.5779154, 0.033306926, -0.03151..."
2,3,Contraband,Red Teenage Melody,Night Lovell,0.801,0.679,4.0,-8.733,0.0,0.0526,...,[contraband],"[red, teenage, melody]",14735,15439,"[0.801, 0.679, 4.0, -8.733, 0.0, 0.0526, 0.17,...","[[0.0211181640625, -0.2275390625, 0.0534667968...","[[0.09716796875, -0.0849609375, 0.271484375, 0...",0,"[-0.280202, -0.15651424, -0.42121008, 0.226249...","[-0.15178716, 0.16047128, -0.513041, -0.028233..."
3,3,Endgame,Out of the Shadow,Rogue Wave,0.281,0.244,7.0,-14.129,0.0,0.0295,...,[endgame],[shadow],21439,17771,"[0.281, 0.244, 7.0, -14.129, 0.0, 0.0295, 0.77...","[[0.1259765625, 0.1103515625, -0.11376953125, ...","[[0.107421875, 0.26953125, -0.0274658203125, -...",0,"[-0.32696834, 0.15048262, 0.02669036, -0.04243...","[-0.29764125, -0.05211623, 0.07484442, 0.01855..."
4,3,Strange to Hear,Naked All the Time,Sports,0.537,0.344,9.0,-14.782,0.0,0.0322,...,"[strange, hear]","[naked, time]",68533,19423,"[0.537, 0.344, 9.0, -14.782, 0.0, 0.0322, 0.84...","[[0.2255859375, 0.07275390625, -0.025512695312...","[[0.0966796875, -0.034423828125, 0.2294921875,...",0,"[-0.3943575, 0.06297159, -0.24746585, 0.327052...","[-0.0279321, 0.29169276, 0.030729515, -0.18929..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1510,453,Can't Fight This Feeling,Wheels Are Turnin',REO Speedwagon,0.408,0.480,9.0,-11.017,1.0,0.0266,...,"[cant, fight, feel]","[wheel, turnin]",11479,17028,"[0.408, 0.48, 9.0, -11.017, 1.0, 0.0266, 0.155...","[[0.2041015625, -0.0303955078125, 0.0006866455...","[[0.13671875, -0.11962890625, -0.06005859375, ...",100,"[0.21349442, 0.6215731, 0.06227052, 0.02683552...","[0.0575988, 0.24469212, 0.16559964, -0.1497041..."
1511,453,Mercy,Rockferry,Duffy,0.793,0.859,0.0,-3.774,1.0,0.0332,...,[mercy],[rockferry],47221,6237,"[0.793, 0.859, 0.0, -3.774, 1.0, 0.0332, 0.266...","[[-0.1044921875, 0.166015625, 0.039306640625, ...","[[1.0793917179107666, 1.7529454231262207, -0.1...",100,"[-0.39874268, 0.16783291, -0.04343432, -0.4568...","[-0.22187924, 0.2762355, 0.068649895, -0.01090..."
1512,453,What Is Life,All Things Must Pass,George Harrison,0.469,0.833,4.0,-5.804,1.0,0.0314,...,[life],"[thing, must, pas]",82164,7973,"[0.469, 0.833, 4.0, -5.804, 1.0, 0.0314, 0.005...","[[-0.06787109375, 0.09521484375, 0.03564453125...","[[0.169921875, 0.049072265625, 0.08154296875, ...",100,"[0.23784855, 0.30514088, 0.031358056, 0.011916...","[-0.01643555, 0.13166331, -0.23213074, -0.1250..."
1513,453,Friends,Friends,Ed Sheeran,0.516,0.158,4.0,-14.433,1.0,0.0672,...,[friend],[friend],25475,6406,"[0.516, 0.158, 4.0, -14.433, 1.0, 0.0672, 0.78...","[[0.07080078125, -0.2138671875, 0.1533203125, ...","[[0.07080078125, -0.2138671875, 0.1533203125, ...",100,"[-0.13863344, 0.24076171, 0.039367624, -0.0360...","[-0.13863344, 0.24076171, 0.039367624, -0.0360..."


In [25]:
I_plus_train.to_parquet("I_plus_train.parquet", index=False)
I_minus_train.to_parquet("I_minus_train.parquet", index=False)
I_plus_val.to_parquet("I_plus_val.parquet", index=False)
I_minus_val.to_parquet("I_minus_val.parquet", index=False)
test.to_parquet("test.parquet", index=False)
#music_pool.to_parquet("music_pool.parquet", index=False)

In [26]:
# Check if all embeddings have the same shape
unique_shapes = I_plus_train['album_Bert_embedding'].apply(lambda x: x.shape).unique()
print("Unique shapes in 'bert_embedding' column:", unique_shapes)


Unique shapes in 'bert_embedding' column: [(768,)]


### Track + Artist + Album

In [ ]:
I_plus_train['AlbumTrackArtist'] = I_plus_train['lemmatized_album_name'] + " - " + I_plus_train['lemmatized_track_name'] + " by " + I_plus_train['artist_name']
I_minus_train['AlbumTrackArtist'] = I_minus_train['lemmatized_album_name'] + " - " + I_minus_train['lemmatized_track_name'] + " by " + I_minus_train['artist_name']
I_plus_val['AlbumTrackArtist'] = I_plus_val['lemmatized_album_name'] + " - " + I_plus_val['lemmatized_track_name'] + " by " + I_plus_val['artist_name']
I_minus_val['AlbumTrackArtist'] = I_minus_val['lemmatized_album_name'] + " - " + I_minus_val['lemmatized_track_name'] + " by " + I_minus_val['artist_name']

In [ ]:
music_pool['AlbumTrackArtist'] = music_pool['lemmatized_album_name'] + " - " + music_pool['lemmatized_track_name'] + " by " + music_pool['artist_name']

In [ ]:
I_plus_train.loc[:,['new_pid','AlbumTrackArtist']]

,new_pid,AlbumTrackArtist
0,0,die alone - get high by Gazebos
1,0,i - nothing gonna hurt baby by Cigarettes Afte...
2,0,head - like by dandelion hands
3,0,candy apple grey - crystal by Hüsker Dü
4,0,say im love - say im love by Banes World
...,...,...
7065,100,mad season - youre go by Matchbox Twenty
7066,100,duncan sheik - barely breathing by Duncan Sheik
7067,100,best air supply one love - make love nothing b...
7068,100,great - deep love by Bee Gees


In [ ]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['AlbumTrackArtist_Bert_embedding'] = I_plus_train['AlbumTrackArtist'].apply(get_bert_embedding)
I_minus_train['AlbumTrackArtist_Bert_embedding'] = I_minus_train['AlbumTrackArtist'].apply(get_bert_embedding)
I_plus_val['AlbumTrackArtist_Bert_embedding'] = I_plus_val['AlbumTrackArtist'].apply(get_bert_embedding)
I_minus_val['AlbumTrackArtist_Bert_embedding'] = I_minus_val['AlbumTrackArtist'].apply(get_bert_embedding)


In [ ]:
music_pool['AlbumTrackArtist_Bert_embedding'] = music_pool['AlbumTrackArtist'].apply(get_bert_embedding)

In [ ]:
# Check if all embeddings have the same shape
unique_shapes = I_plus_train['AlbumTrackArtist_Bert_embedding'].apply(lambda x: x.shape).unique()
print("Unique shapes in 'bert_embedding' column:", unique_shapes)


Unique shapes in 'bert_embedding' column: [(768,)]


In [ ]:
I_plus_train.to_pickle("I_plus_train_with_bert_albumtrackartist.pkl")
I_minus_train.to_pickle("I_minus_train_with_bert_albumtrackartist.pkl")
I_plus_val.to_pickle("I_plus_val_with_bert_albumtrackartist.pkl")
I_minus_val.to_pickle("I_minus_val_with_bert_albumtrackartist.pkl")
music_pool.to_pickle("music_pool_with_bert_albumtrackartist.pkl")